# Глава 13. Интерпретируемость

[не проверено]

## Введение
Языковые модели переводят тексты, пишут код и решают олимпиадные задачи, но мы до сих пор плохо понимаем, как именно они это делают. Архитектуру, функцию потерь и данные мы выбрали сами, а вот алгоритм, который модель выучила, остаётся закрытым. Крис Ола, автор термина, сравнивает обучение с компилятором, который сам пишет программу и сразу превращает её в машинный код, то есть в веса [(Olah, 2022)](https://transformer-circuits.pub/2022/mech-interp-essay/index.html). Механистическая интерпретируемость (mechanistic interpretability) пытается провести реверс-инжиниринг такой программы: восстановить по весам и активациям понятные человеку алгоритмы.

Зачем это нужно? Для отладки: понять причину ошибки, а не досыпать данных в надежде, что она исчезнет. Для безопасности: бенчмарки (см. главу 8) проверяют поведение на тестовых примерах, но если модель, скажем, ведёт себя иначе, когда понимает, что её проверяют, поведенческий тест этого не заметит. И из научного интереса: языковые модели — первые системы, владеющие языком на уровне человека, и, в отличие от мозга, нам доступна каждая их активация.

Классические методы объяснимого ИИ атрибутируют ответ ко входу: градиентные карты значимости [(Simonyan et al., 2013)](https://arxiv.org/abs/1312.6034), интегрированные градиенты [(Sundararajan et al., 2017)](https://arxiv.org/abs/1703.01365), LIME [(Ribeiro et al., 2016)](https://arxiv.org/abs/1602.04938), SHAP [(Lundberg & Lee, 2017)](https://arxiv.org/abs/1705.07874). Они говорят, какие слова повлияли на ответ, но не говорят, как модель к нему пришла. Механистический подход смотрит внутрь: какие представления формирует модель, какие компоненты их вычисляют и как эти компоненты связаны.

Направление выросло из компьютерного зрения. В 2020 году в серии статей Circuits на платформе Distill [(Olah et al., 2020)](https://distill.pub/2020/circuits/zoom-in/) разобрали нейроны сети InceptionV1 и нашли среди них детекторы кривых и частей объектов, связанные в понятные схемы. Год спустя команда Anthropic перенесла подход на Трансформеры [(Elhage et al., 2021)](https://transformer-circuits.pub/2021/framework/index.html). Всю область можно свести к трём вопросам: что модель представляет (признаки), как она вычисляет (схемы) и как убедиться, что объяснение верно, а не просто правдоподобно. Им и посвящена глава.

## Трансформер глазами интерпретатора
В главе 4 мы описывали Трансформер как стопку одинаковых блоков. Для интерпретации удобнее другой, математически эквивалентный взгляд [(Elhage et al., 2021)](https://transformer-circuits.pub/2021/framework/index.html), в котором вычисление раскладывается на сумму простых слагаемых.

### Остаточный поток
Ни один слой не перезаписывает вектор токена, а только прибавляет к нему свой результат. Если развернуть рекурсию, финальное состояние окажется суммой вкладов всех компонентов:

$$x^{(L)} = x^{(0)} + \sum_{l,h} \mathrm{head}_{l,h} + \sum_{l} \mathrm{MLP}_l, \qquad \text{logits} = W_U\, \mathrm{LN}\big(x^{(L)}\big),$$

где $x^{(0)}$ — эмбеддинг токена, а $W_U$ — матрица анэмбеддинга. Этот вектор называют остаточным потоком (residual stream). Его удобно представлять как доску в переговорной: каждый участник (голова внимания или MLP) читает с доски то, что ему нужно, и дописывает свои заметки, ничего не стирая, а в конце анэмбеддинг читает всё накопленное и выдаёт прогноз. Читают и пишут компоненты линейными проекциями, поэтому вклад любого из них можно проследить. При этом доска тесная: у GPT-2 small размерность потока 768, а пишут в него 144 головы внимания и 12 слоёв MLP.

[рисунок: схема остаточного потока по мотивам Elhage et al., 2021]

### QK- и OV-схемы
Матрицы головы внимания входят в формулу только попарными произведениями:

$$A_{ij} = \mathrm{softmax}_j\!\left(\frac{x_i^\top W_{QK}\, x_j}{\sqrt{d_{\text{head}}}}\right), \qquad \mathrm{head}(x)_i = \sum_{j \le i} A_{ij}\, W_{OV}\, x_j,$$

где $W_{QK} = W_Q^\top W_K$ и $W_{OV} = W_O W_V$. Получаются две независимые схемы. QK-схема решает, куда смотреть («к кому обратиться»), OV-схема — что перенести и как записать («что у него взять»). Обе матрицы низкого ранга, то есть каждая голова работает в небольшом подпространстве потока. Домножив их на эмбеддинги, получим виртуальные веса в пространстве словаря: например, у копирующей головы матрица $W_U W_{OV} W_E$ близка к диагональной — посмотрев на токен «Москва», голова повышает логит того же токена.

Так модель раскладывается на пути. Без единого слоя остаётся прямой путь $W_U W_E$, и это ровно биграммная модель из главы 1. Однослойная модель добавляет к нему skip-триграммы вида «$a \dots b \to c$»: голова смотрит на ранее встреченный токен $a$ и повышает вероятность $c$ после текущего $b$. Со вторым слоем появляется композиция: головы второго слоя читают то, что записали головы первого, через запросы (Q-композиция), ключи (K-композиция) или значения (V-композиция). Самый известный пример K-композиции — индукционные головы, о которых пойдёт речь в разделе о схемах.

### MLP и нормализация
Полносвязный слой можно переписать как сумму вкладов отдельных нейронов:

$$\mathrm{MLP}(x) = \sum_{i} \sigma\big(k_i^\top x + b_i\big)\, v_i,$$

где $k_i$ — строка входной матрицы, а $v_i$ — столбец выходной. [(Geva et al., 2020)](https://arxiv.org/abs/2012.14913) предложили читать это как память «ключ–значение»: $k_i$ задаёт паттерн, на который реагирует нейрон, а $v_i$ — то, что он добавит в поток. Если спроецировать $v_i$ через $W_U$, видно, вероятность каких токенов нейрон повышает [(Geva et al., 2022)](https://arxiv.org/abs/2203.14680).

Линейный анализ портит нормализация: деление на норму вектора нелинейно. Обычно её «замораживают», беря коэффициент из реального прогона, и тогда путь от компонента до логитов становится линейным. А [(Heimersheim, 2024)](https://arxiv.org/abs/2409.13710) показал, что GPT-2 можно дообучить вовсе без LayerNorm почти без потери качества, что удобно для экспериментов.

## Признаки и суперпозиция
Признак (feature) — это свойство входа, которое модель кодирует в активациях: «текст на французском», «токен внутри кавычек», «речь идёт об Эйфелевой башне». Строгого определения у признака нет, и вполне возможно, что модель пользуется признаками, для которых у нас нет названия.

### Полисемантичность
Самая простая гипотеза — один нейрон кодирует один признак. Иногда она подтверждается. [(Karpathy et al., 2015)](https://arxiv.org/abs/1506.02078) нашли в символьной LSTM ячейку, которая отслеживает, открыта ли кавычка. [(Radford et al., 2017)](https://arxiv.org/abs/1704.01444) обнаружили «нейрон тональности» в модели, обученной на отзывах Amazon: фиксируя его значение, можно было заставить модель писать восторженные или разгромные отзывы. Но чаще нейроны полисемантичны (polysemantic), то есть реагируют на несколько не связанных понятий. В InceptionV1 нашёлся нейрон, который срабатывает одновременно на морды кошек, передние части автомобилей и кошачьи лапы [(Olah et al., 2020)](https://distill.pub/2020/circuits/zoom-in/). В языковых моделях полисемантичность повсеместна [(Gurnee et al., 2023)](https://arxiv.org/abs/2305.01610). Объяснений два: признаки модели — это направления, не совпадающие с нейронами, и признаков больше, чем нейронов.

### Гипотеза линейных представлений
С линейной структурой представлений мы встречались в главе 2: $\vec{\text{король}} - \vec{\text{мужчина}} + \vec{\text{женщина}} \approx \vec{\text{королева}}$ [(Mikolov et al., 2013)](https://aclanthology.org/N13-1090/). Гипотеза линейных представлений (linear representation hypothesis) обобщает это наблюдение: признаки соответствуют направлениям в пространстве активаций, а выраженность признака — проекции на это направление [(Park et al., 2023)](https://arxiv.org/abs/2311.03658). Подтверждений много. Линейные пробы восстанавливают по активациям Llama-2 координаты городов и даты событий, так что по их предсказаниям можно нарисовать карту мира [(Gurnee & Tegmark, 2023)](https://arxiv.org/abs/2310.02207). Истинные и ложные утверждения разделяются одним направлением, и сдвиг вдоль него меняет оценку модели [(Marks & Tegmark, 2023)](https://arxiv.org/abs/2310.06824). Логика простая: компоненты читают поток линейными проекциями, поэтому нужный признак выгоднее всего сделать доступным одной проекцией.

Есть и исключения. Дни недели и месяцы в GPT-2 и Mistral лежат на окружности в двумерной плоскости, и модель пользуется этой геометрией, отвечая на вопросы вроде «какой день будет через два дня после пятницы» [(Engels et al., 2024)](https://arxiv.org/abs/2405.14860). Поэтому гипотезу чаще формулируют мягче: признаки — это подпространства малой, чаще всего единичной, размерности.

### Суперпозиция
Понятий, которые нужны модели, заведомо больше, чем измерений в потоке. Но в многомерном пространстве почти ортогональных векторов помещается экспоненциально больше, чем измерений. Платой становится интерференция: активный признак немного «просвечивает» в проекциях на чужие направления. Это терпимо, если признаки разрежены, то есть одновременно активна лишь малая их часть, а язык как раз такой: в каждом тексте речь идёт о немногих темах, людях и местах. Способность кодировать больше признаков, чем измерений, называют суперпозицией (superposition).

Её изучили на игрушечной модели [(Elhage et al., 2022)](https://arxiv.org/abs/2209.10652): $n$ разреженных признаков с важностями $I_i$ сжимаются в $m < n$ измерений и восстанавливаются обратно:

$$h = W x, \qquad x' = \mathrm{ReLU}\big(W^\top h + b\big), \qquad \mathcal{L} = \mathbb{E}_x \sum_i I_i\, (x_i - x'_i)^2$$

Пока признаки плотные, модель, подобно методу главных компонент, хранит только $m$ самых важных и игнорирует остальные. С ростом разреженности она скачком переходит к суперпозиции и упаковывает признаки в правильные фигуры: антиподальные пары, треугольники, пятиугольники. Интерференцию при этом отсекает ReLU с отрицательным смещением, поэтому без нелинейности суперпозиция невыгодна. Отсюда естественное объяснение полисемантичности: если признаков больше, чем нейронов, каждый нейрон вынужден участвовать в кодировании нескольких.

[рисунок: фазовая диаграмма и геометрия признаков из Elhage et al., 2022]

### Привилегированный базис
Почему вообще стоит смотреть на отдельные нейроны MLP, но не на отдельные координаты остаточного потока? Если повернуть поток ортогональной матрицей $R$, домножив на неё все читающие и пишущие матрицы, модель вычислит ту же самую функцию. Значит, у координат потока нет особого смысла, или, как говорят, у него нет привилегированного базиса (privileged basis). У нейронов MLP он есть: функция активации применяется покоординатно, и $\sigma(Rz) \neq R\,\sigma(z)$. На практике оптимизатор Adam всё же создаёт в потоке выделенные координаты с аномально большими значениями [(Elhage et al., 2023)](https://transformer-circuits.pub/2023/privileged-basis/index.html), но это скорее исключение.

Итог раздела: признаки — это направления, их больше, чем измерений, и с осями они почти не совпадают. Если мы знаем, что ищем, помогут пробы. Если не знаем, придётся восстанавливать словарь признаков без учителя с помощью разреженных автоэнкодеров.

## Наблюдательные методы
Методы интерпретации, как и в нейробиологии, делятся на две группы: наблюдение за активностью и вмешательство в неё. Наблюдательные методы дёшевы и хорошо подходят для выдвижения гипотез, но находят лишь корреляции: то, что информация в активациях есть, ещё не значит, что модель ею пользуется.

### Анализ нейронов
Самый прямой вариант анализа - посмотреть тепловую карту активаций.

[(Voita et al., 2023)](https://arxiv.org/abs/2309.04827) исследовали активации в архитекутуре моделей OPT. Несколько важных находок для понимания работы моделей. Во-первых, большинство нейронов ни разу не активировались. Особенно в ранних слоях (до 70% в отдельных слоях). то есть модель избыточна. Во-вторых, многие нейроны в ранних слоях работают как детекторы конкретных токенов или n-грамм, а заодно подавляют предсказание этих же токенов. В-третьих, нашли чисто позиционные нейроны, чья активация зависит только от позиции токена, а не от его содержания.

[(Dai et al., 2021)](https://arxiv.org/abs/2104.08696) обнаружили в BERT моделях наличие то, что они назвали нейронами "знания", которые находятся в FFN-слоях и отвечают за хранение фактические информации. Для этого они задавали вопросы на знание («Столица Франции — [MASK]») и выделяли нейроны, сильнее всего влияющие на правильный ответ (отключение портит ответ). Этот же принцип лег в основу техники точечного редактирования знаний в моделях - можно занулить нейрон, и модель забудет ассоциированный с ним факт.

Метод Dai правда быстро раскритиковали. В качестве контрпримера [(Niu et al., 2024)](https://arxiv.org/abs/2405.02421) замаскировали обычные слова, не факты, и FFN нейроны активировались на них ничуть не меньше. То есть на самом деле нейроны реагировали больше на синтаксическую согласованность, чем на необходимость достать конкретный факт. Кроме того, редактирование через зануление этих нейронов работает слабо. Фактическая информация действительно хранится в FFN, но не концентрировано в отдельных нейронах а все таки распределённо.

Также были попытки по карте акиваций выделить общий паттерн - смотрим, когда нейрон сильнее всего срабатывает, определяем признак и ассоциируем с ним. Её также подвергли критике. []() провели эксперимент, на нескольких корпусах текстов. На каждом корпусе это были совершенно разные запросы.

### Пробинг
Большее распространение получил противоположном направлении. Не по активациям искать паттерны, а по паттерна.
Выбираем некоторое свойство (например, "тональность высказывания" или "пол главного субъекта"), поверх активаций анализиуруемого нейрона присоединить небольшой классификатор. Веса модели замораживаются, а этот классификатор обучается определять наличие данного свойства. [(Alain & Bengio, 2016)](https://arxiv.org/abs/1610.01644) назвали такой классификатор зондом (Probe). Высокая точность пробинга означает, что сигнал в активациях есть. Применять пробинг можно как к отдельному токену, или к фрагменту текста - тогда делается усредение.

Перебирая различные свойства, можно построить довольно детальную карту, что где происходит в Трансформере. Так [(Tenney et al., 2019)](https://arxiv.org/abs/1905.05950) выяснили, что в BERT моделях стабильно наблюдается четкая иерархия признаков: части речи распознаются в нижних слоях, синтаксис в средних, а сущностные связи (coreferences) в самых верхних. [(Hewitt & Manning, 2019)](https://aclanthology.org/N19-1419/) выяснили, что по активациям можно даже восстанавливать синтаксические деревья - чем ближе токены в дереве, тем ближе их активации.

У пробинга есть две проблемы. Во-первых, слишком многослойный классификатор может выучить задачу, поэтому её сравнивают с контрольной задачей со случайными метками и смотрят на разницу точностей [(Hewitt & Liang, 2019)](https://arxiv.org/abs/1909.03368). Во-вторых, пробинг детектирует наличие сигнала, но не его фактическое использование для генерации ответа. Чтобы проверить, нужна ли информация модели, нейроны зануляют и смотрят: как меняется точность предсказания. [(Elazar et al., 2020)](https://arxiv.org/abs/2006.00995) такую механику назвали амнезический пробинг.

### Logit Lens
Идея метода - подать активации с каждого промежуточного слоя сразу на последнюю проекцию и посмотреть, как юудет меняться выходное распределение. [(Nostalgebraist, 2020)](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) назвали этот приём Logit Lens, логиты анализируются под микроскопом. На примере модели GPT-2 авторы показали, как прогноз постепенно уточняется по слоям, а для простых токенов складывается уже к середине сети. 

Проблема метода в том что представления промежуточных слоёв «дрейфуют», и logit lens выдаёт шум. Модификация метода Tuned lens дополниетльно обучая для каждого слоя небольшое аффинное преобразование в пространство последнего слоя [(Belrose et al., 2023)](https://arxiv.org/abs/2303.08112). С помощью Logit Lens, например, [(Wendler et al., 2024)](https://arxiv.org/abs/2402.10588) выяснили, что модель Llama-2 при переводе с французского на китайский в средних слоях "думает" английскими словами.

### Паттерны внимания
Веса внимания соблазнительно считать объяснением: их легко нарисовать, а в BERT нашлись головы, отслеживающие синтаксические отношения [(Clark et al., 2019)](https://arxiv.org/abs/1906.04341). Но вокруг них развернулся спор. [(Jain & Wallace, 2019)](https://arxiv.org/abs/1902.10186) показали, что совсем другие распределения внимания часто дают тот же ответ, а [(Wiegreffe & Pinter, 2019)](https://arxiv.org/abs/1908.04626) возразили, что вывод зависит от того, что считать объяснением. С механистической точки зрения ограничения понятны. Паттерн описывает только QK-схему и молчит о том, что голова переносит: большой вес на служебном токене обычно сопровождается почти нулевым вкладом [(Kobayashi et al., 2020)](https://arxiv.org/abs/2004.10102). Первый токен часто служит «стоком» внимания, куда голова сбрасывает вес, когда ей нечего искать [(Xiao et al., 2023)](https://arxiv.org/abs/2309.17453). Поэтому паттерны хороши для гипотез, но не для доказательств.

### Прямая атрибуция к логитам
Раз остаточный поток — сумма вкладов, логит можно разложить по компонентам. Заморозив нормализацию с коэффициентом $\sigma$, получим вклад компонента $c$ в выбор токена $t_1$ вместо $t_2$:

$$\mathrm{DLA}(c) = \frac{\big(W_U[t_1] - W_U[t_2]\big) \cdot c}{\sigma}$$

Например, в предложении «Когда Мэри и Джон пошли в магазин, Джон дал напиток» (в оригинале на английском) так находят головы, которые сильнее всего толкают модель к ответу «Мэри». Прямая атрибуция (direct logit attribution) считается за один прогон, но видит только прямой эффект: голова, которая направляет внимание другой головы, но сама в логиты не пишет, останется незамеченной. Чтобы измерить полный эффект, нужно вмешиваться.

## Каузальные интервенции
Джуда Перл различает наблюдение $P(Y \mid X = x)$ и вмешательство $P(Y \mid \mathrm{do}(X = x))$ [(Pearl & Mackenzie, 2018)](https://en.wikipedia.org/wiki/The_Book_of_Why): продажи мороженого коррелируют с числом утоплений, но запрет мороженого утоплений не уменьшит. Для нейросети вмешательство дёшево: можно заменить любую активацию и повторить прогон, о чём нейробиолог может только мечтать. Одной из первых систематических работ стала [(Vig et al., 2020)](https://arxiv.org/abs/2004.12265), где к GPT-2 применили каузальный анализ медиации из эпидемиологии. Меняем вход и сравниваем полный эффект на выход с эффектом, который проходит через отдельный компонент $z$:

$$\mathrm{TE} = M(x') - M(x), \qquad \mathrm{IE}_z = M\big(x;\; z \leftarrow z(x')\big) - M(x),$$

где $M$ — интересующая нас метрика. Все методы этого раздела — вариации этой идеи.

### Абляции
Самое простое вмешательство — выключить компонент и посмотреть, как изменится метрика. Вопрос в том, чем его заменить: нулём (zero ablation), средним значением по датасету (mean ablation), значением с другого случайного примера (resample ablation) или константой, минимизирующей потери модели [(Li & Janson, 2024)](https://arxiv.org/abs/2409.09951). Аналогия: чтобы оценить вклад сотрудника, можно оставить его стул пустым, посадить на его место «среднего сотрудника» или коллегу из другого проекта. Пустой стул — худший вариант: команда может развалиться не потому, что сотрудник важен, а потому, что все привыкли, что стул занят. Нулевой вектор тоже часто выводит модель за пределы распределения, поэтому чаще используют среднее или ресэмплинг.

Есть и вторая сложность: модели умеют себя чинить. Если выключить слой внимания, последующие слои частично берут его работу на себя. Этот эффект назвали эффектом гидры [(McGrath et al., 2023)](https://arxiv.org/abs/2307.15771). Маленький эффект абляции ещё не значит, что компонент не важен.

### Патчинг активаций
Патчинг активаций (activation patching, или causal tracing) отвечает на вопрос, через какие компоненты проходит нужная информация. Берут пару минимально различающихся входов: чистый (clean) вызывает интересующее поведение, испорченный (corrupted) — нет. Например, «Когда Мэри и Джон пошли в магазин, Джон дал напиток» → «Мэри» и тот же текст с тремя другими именами. Прогоняем испорченный вход, но в одном месте подставляем активацию из чистого прогона и смотрим, насколько восстановилось поведение. Перебрав слои, позиции и компоненты, получаем карту важности. Восстановление измеряют разностью логитов правильного и неправильного ответа, нормированной на её значения в чистом и испорченном прогонах: в отличие от вероятности, она не насыщается [(Zhang & Nanda, 2023)](https://arxiv.org/abs/2309.16042).

[рисунок: тепловая карта патчинга по слоям и позициям]

Метод прославила работа ROME [(Meng et al., 2022)](https://arxiv.org/abs/2202.05262). Авторы портили промпт «Эйфелева башня находится в городе», зашумляя эмбеддинги субъекта, и выяснили, что факт достают MLP средних слоёв на последнем токене субъекта, а внимание верхних слоёв переносит его к позиции ответа. На этой основе они предложили редактировать факты точечным изменением весов одного слоя MLP. Продолжение оказалось поучительным: [(Hase et al., 2023)](https://arxiv.org/abs/2301.04213) показали, что редактирование слоёв, где факт якобы не хранится, работает не хуже. Локализация в смысле патчинга и «место, где удобно вмешаться», — разные вещи.

Важно и направление патчинга [(Heimersheim & Nanda, 2024)](https://arxiv.org/abs/2404.15255). Подставляя чистые активации в испорченный прогон (denoising), мы ищем компоненты, которых достаточно для поведения. Подставляя испорченные в чистый (noising), ищем необходимые. Результаты могут расходиться. Если ответ требует одновременной работы двух голов (логическое И), денойзинг по отдельности не найдёт ни одной из них. Если хватает любой из двух (логическое ИЛИ, например основная и запасная), ни одной не найдёт нойзинг. Поэтому используют оба варианта.

### Патчинг путей и атрибуционный патчинг
Обычный патчинг измеряет полный эффект компонента. Чтобы восстановить схему, нужно знать связи: какая голова влияет на какую. Патчинг путей (path patching) пропускает изменение только по ребру от отправителя к получателю, замораживая все остальные пути [(Goldowsky-Dill et al., 2023)](https://arxiv.org/abs/2304.05969).

Оба метода дороги: на каждое место нужен отдельный прогон. Нил Нанда предложил заменить патчинг линейным приближением — атрибуционным патчингом (attribution patching) [(Nanda, 2023)](https://www.neelnanda.io/mechanistic-interpretability/attribution-patching):

$$M\big(x_{\text{corr}};\; a \leftarrow a_{\text{clean}}\big) - M(x_{\text{corr}}) \approx \big(a_{\text{clean}} - a_{\text{corr}}\big)^\top \left.\frac{\partial M}{\partial a}\right|_{x_{\text{corr}}}$$

Чтобы посчитать это сразу для всех активаций, хватает двух прямых прогонов и одного обратного. Приближение ломается там, где модель резко нелинейна, например при насыщенном softmax внимания, и улучшенная версия AtP* исправляет основные такие случаи [(Kramár et al., 2024)](https://arxiv.org/abs/2403.00745). На практике атрибуционный патчинг используют как дешёвый фильтр, а отобранных кандидатов перепроверяют обычным патчингом.

### Каузальные абстракции
Конечная цель — не список важных компонентов, а алгоритм. Теория каузальных абстракций [(Geiger et al., 2021)](https://arxiv.org/abs/2106.02997) предлагает формулировать гипотезу как маленький граф с понятными переменными и проверять, что вмешательство в соответствующее представление модели действует так же, как вмешательство в переменную. Пусть гипотеза гласит, что при сложении $a + b + c$ модель сначала вычисляет $s = a + b$. Подставим представление $s$ из прогона на $4 + 3 + 9$ в прогон на $1 + 2 + 5$. Если гипотеза верна, модель ответит $7 + 5 = 12$, а не 8 и не 16.

Переменная может быть распределена по подпространству, не совпадающему с нейронами. Метод DAS обучает ортогональный поворот $R$ и вмешивается только в первые $k$ координат повёрнутого базиса, подбирая $R$ так, чтобы ответы совпадали с предсказаниями гипотезы [(Geiger et al., 2023)](https://arxiv.org/abs/2303.02536). По сути это пробинг с каузальным критерием качества. Но гибкость опасна: найденное подпространство может «будить» спящий путь, который в обычной работе модели не участвует [(Makelov et al., 2023)](https://arxiv.org/abs/2311.17030). Ещё строже проверяет гипотезы метод causal scrubbing [(Chan et al., 2022)](https://www.lesswrong.com/posts/JvZhhzycHu2Yd57RN/causal-scrubbing-a-method-for-rigorously-testing): всё, что гипотеза объявляет неважным, заменяется ресэмплингом, и если гипотеза полна, качество модели не падает.

Для таких экспериментов есть готовые библиотеки: [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens), nnsight [(Fiotto-Kaufman et al., 2024)](https://arxiv.org/abs/2407.14561) и pyvene [(Wu et al., 2024)](https://arxiv.org/abs/2403.07809).

## Схемы
[супер интересно]

Схема (circuit) — это подграф модели, который реализует конкретное поведение.

Хорошая схема должна быть достоверной (если оставить только её, поведение сохранится), полной (вне её нет ничего важного для поведения) и минимальной (в ней нет лишнего).

### Индукционные головы
Решили посмотреть, как именно работает внимание в специфических ситуациях генерации проолжения.
Первым кейсом стал In-context llearning. Продолжение при начлии конектста. подробно разобранной схемой в Трансформерах стали индукционные головы [(Olsson et al., 2022)](https://arxiv.org/abs/2209.11895). Если в тексте уже встречалась пара токенов $[a][b]$, то, увидев $a$ снова, модель предсказывает $b$: «В книге Анна Каренина. Имя героини Анна ...» → «Каренина».

Оказывается, у голов самовнимания появляется специализация. Авторы выделяют два типа голов. Первый назвали previous token head, которая работает просто как функция `lag()` - она запоминает предыдуший токен. Второй - induction head, которая ищет в предыдщуем тексте тот же префикс A и запоминает его разрешение B. Это разрешегние B и идет в продолжение.

Любопытно, что индукционные головы появляются скачком: в определённый момент обучения на кривой потерь видна ступенька, и одновременно резко улучшается обучение в контексте (in-context learning). Авторы приводят ряд аргументов в пользу того, что индукционные головы — один из главных механизмов этой способности.

[рисунок: паттерн внимания индукционной головы на повторяющейся последовательности]

### Схема IOI
Далее [(Wang et al., 2022)](https://arxiv.org/abs/2211.00593) решили специфицировать задачу и посомтреть, как модель выбирает косвенное дополнение задача IOI (indirect object identification). В предложении «Когда Мэри и Джон пошли в магазин, Джон дал напиток ...» модель должна назвать Мэри, не Джон.

Использовали GPT-2 с 144 головами самовнимания. Удалось найти схему из 26 голов и имеющих узкую специализацию. К двум головам, которые уже рассматривали, "Previous Token head" и "Induction Head" добавились еще 4. Головы дубликатов (Duplicate Token head) замечают, что «Джон» уже встречался. Головы S-ингибиции (Subject Inhibition) подавляют внимание к повторившемуся имени. Головы перемещения имён (name movers) смотрят на оставшееся имя и копируют его в ответ. Нашлись и запасные головы, которые берут работу на себя, если выключить основные, и головы, которые, наоборот, снижают уверенность модели.

### Другие схемы
[(Hanna et al., 2023)](https://arxiv.org/abs/2305.00586) разобрали, как GPT-2 продолжает фразу «Война длилась с 1732 по 17..», выбирая только годы больше 32. [(Nanda et al., 2023)](https://arxiv.org/abs/2301.05217) полностью восстановили алгоритм маленького Трансформера, выучившего сложение по модулю: числа представлялись поворотами на окружности, а сложение выполнялось через тригонометрические тождества (эту работу мы упоминали в главе 5 в связи с гроккингом). [(Lieberum et al., 2023)](https://arxiv.org/abs/2307.09458) проверили, масштабируется ли анализ схем, на модели Chinchilla с 70 млрд параметров: ключевые головы для ответа на вопрос с вариантами найти удалось, но их объяснение лишь частично переносилось на другие формулировки задачи.

### Автоматический поиск схем
Ручной поиск подходящих схем трудоемкий. Метод ACDC [(Conmy et al., 2023)](https://arxiv.org/abs/2304.14997) автоматизирует его: идёт от выхода ко входу и удаляет из графа рёбра, патчинг которых почти не меняет метрику. Edge attribution patching [(Syed et al., 2023)](https://arxiv.org/abs/2310.10348) заменяет перебор градиентным приближением и работает на порядки быстрее. Сравнивать методы поиска схем помогает бенчмарк MIB [(Mueller et al., 2025)](https://arxiv.org/abs/2504.13151).

У анализа схем есть ограничения. Каждую схему ищут под конкретную задачу и конкретный набор промптов. Самовосстановление искажает оценки важности. А главное, на уровне голов и нейронов схемы получаются запутанными из-за суперпозиции. Следующий шаг — перейти от компонентов к признакам.

## Разреженные автоэнкодеры
Если признаков больше, чем измерений, и они хранятся в суперпозиции, нужно разложить активации на большой набор направлений так, чтобы на каждом входе было активно лишь несколько из них. Это задача словарного обучения (dictionary learning). Для эмбеддингов слов её решали ещё [(Arora et al., 2018)](https://arxiv.org/abs/1601.03764), показав, что вектор многозначного слова раскладывается на векторы его отдельных значений. В 2023 году две группы независимо применили к активациям языковых моделей разреженный автоэнкодер (sparse autoencoder, SAE) [(Cunningham et al., 2023)](https://arxiv.org/abs/2309.08600), [(Bricken et al., 2023)](https://transformer-circuits.pub/2023/monosemantic-features/index.html):

$$f = \mathrm{ReLU}\big(W_{enc}\, x + b_{enc}\big), \qquad \hat{x} = W_{dec}\, f + b_{dec}, \qquad \mathcal{L} = \|x - \hat{x}\|^2 + \lambda \|f\|_1$$

Размерность $f$ в десятки и сотни раз больше размерности $x$, а штраф $L_1$ заставляет большинство координат $f$ быть нулями. Столбцы $W_{dec}$ образуют словарь признаков, а $f$ — их активации. Получается то, чего не дают нейроны: признаки, как правило, моносемантичны. На однослойной модели нашлись признаки арабского письма, последовательностей ДНК, кодировки base64 и тысячи других.

[рисунок: разреженный автоэнкодер над остаточным потоком]

### Варианты и метрики
Штраф $L_1$ систематически занижает величину активаций, поэтому появились архитектурные варианты. TopK SAE просто оставляет $k$ наибольших активаций [(Gao et al., 2024)](https://arxiv.org/abs/2406.04093), а JumpReLU SAE разделяют решение «активен ли признак» и оценку его величины [(Rajamanoharan et al., 2024)](https://arxiv.org/abs/2407.14435). Качество SAE оценивают по нескольким осям: среднее число активных признаков $L_0$, доля объяснённой дисперсии, рост функции потерь модели при подстановке $\hat{x}$ вместо $x$ и интерпретируемость признаков. Эти метрики сводит вместе бенчмарк SAEBench [(Karvonen et al., 2025)](https://arxiv.org/abs/2503.09532).

С ростом словаря признаки расщепляются (feature splitting): условный признак «математика» распадается на «алгебру», «геометрию» и так далее. Бывает и обратное явление, поглощение (feature absorption): общий признак «слово начинается на S» молчит на слове «short», потому что его работу забрал себе специфичный признак этого слова [(Chanin et al., 2024)](https://arxiv.org/abs/2409.14507).

### Масштабирование
В 2024 году SAE обучили на больших моделях. Anthropic получила до 34 млн признаков для Claude 3 Sonnet [(Templeton et al., 2024)](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html). Признаки оказались абстрактными, мультиязычными и мультимодальными: признак моста Golden Gate активировался и на упоминания моста на разных языках, и на его фотографии. Нашлись и признаки, важные для безопасности: обман, угодничество, уязвимый код. OpenAI обучила SAE с 16 млн признаков на активациях GPT-4 [(Gao et al., 2024)](https://arxiv.org/abs/2406.04093), а Google DeepMind выпустила открытый набор SAE для всех слоёв моделей Gemma 2 [(Lieberum et al., 2024)](https://arxiv.org/abs/2408.05147). Найденные признаки удобно изучать на платформе [Neuronpedia](https://www.neuronpedia.org).

### Транскодеры и кросскодеры
SAE раскладывает активации в одной точке, но ничего не говорит о вычислении между точками. Транскодер (transcoder) учится не восстанавливать вход MLP, а предсказывать его выход через разреженный слой, то есть заменяет MLP интерпретируемым аналогом [(Dunefsky et al., 2024)](https://arxiv.org/abs/2406.11944). Кросскодер (crosscoder) читает и восстанавливает сразу несколько слоёв или даже две модели, что позволяет, например, найти признаки, появившиеся при дообучении базовой модели в чат-модель [(Lindsey et al., 2024)](https://transformer-circuits.pub/2024/crosscoders/index.html).

### Критика
SAE решили не все проблемы. Часть дисперсии они не восстанавливают, и в этой «тёмной материи» может прятаться важное [(Engels, Riggs & Tegmark, 2024)](https://arxiv.org/abs/2410.14670). Словарь зависит от размера и даже от случайной инициализации, поэтому неясно, находит ли SAE «настоящие» признаки модели или удобное приближение. Наконец, на практических задачах SAE пока часто проигрывают простым методам: например, линейные пробы на сырых активациях, как правило, не уступают пробам на признаках SAE [(Kantamneni et al., 2025)](https://arxiv.org/abs/2502.16681).

## Графы атрибуции
Следующий шаг — соединить признаки в графы вычислений. Предшественником стала работа о разреженных схемах признаков [(Marks et al., 2024)](https://arxiv.org/abs/2403.19647), а в 2025 году Anthropic предложила метод отслеживания схем (circuit tracing) [(Ameisen et al., 2025)](https://transformer-circuits.pub/2025/attribution-graphs/methods.html). Все MLP модели заменяются межслойными транскодерами, которые пишут сразу в несколько последующих слоёв, и получается замещающая модель (replacement model), где вычисление проходит через интерпретируемые признаки. Для конкретного промпта паттерны внимания и нормализацию фиксируют, после чего связи между признаками становятся линейными, и вклад каждого признака в каждый следующий считается точно. Получается граф атрибуции (attribution graph): узлы — активные признаки, рёбра — их влияния друг на друга. Ошибку транскодеров добавляют в граф отдельными узлами, чтобы честно показать, какая часть вычислений осталась необъяснённой.

[рисунок: граф атрибуции для вопроса о столице штата, где находится Даллас]

С помощью таких графов авторы разобрали десяток поведений модели Claude 3.5 Haiku [(Lindsey et al., 2025)](https://transformer-circuits.pub/2025/attribution-graphs/biology.html). На вопрос о столице штата, где находится Даллас, модель сначала активирует признак «Техас» и уже из него получает «Остин», то есть действительно рассуждает в два шага. Сочиняя рифмованную строку, модель заранее выбирает слово для рифмы и строит строку так, чтобы к нему прийти; если подавить этот признак, строка заканчивается другой рифмой. При сложении двузначных чисел работают два параллельных пути: один грубо оценивает величину суммы, другой точно вычисляет последнюю цифру. При этом, если спросить модель, как она считала, она описывает школьный алгоритм с переносом. Галлюцинации объясняются так: по умолчанию в модели активна схема «я не знаю ответа», а признаки знакомой сущности её подавляют. Если имя кажется знакомым, но фактов о нём модель не знает, подавление срабатывает ошибочно.

Ограничения метода авторы перечисляют сами: граф описывает один промпт, объясняет лишь часть вычислений и плохо показывает, почему внимание смотрит туда, куда смотрит. Инструменты для построения графов на открытых моделях опубликованы в библиотеке circuit-tracer вместе с интерфейсом на Neuronpedia [(Anthropic, 2025a)](https://www.anthropic.com/research/open-source-circuit-tracing).

## Управление через представления
Если признаки — это направления, их можно не только читать, но и добавлять. Самый простой способ получить такое направление — взять разность средних активаций на двух контрастных наборах промптов и добавить её в остаточный поток с коэффициентом $\alpha$:

$$v = \mathbb{E}\big[h_l(x) \mid x \in \mathcal{D}^{+}\big] - \mathbb{E}\big[h_l(x) \mid x \in \mathcal{D}^{-}\big], \qquad h_l \leftarrow h_l + \alpha\, v$$

Вектор $v$ называют вектором управления (steering vector). Уже разность активаций для слов «Love» и «Hate», добавленная в поток, меняет тональность генерации [(Turner et al., 2023)](https://arxiv.org/abs/2308.10248). Метод CAA усредняет такие разности по многим парам ответов и управляет, например, угодничеством модели [(Rimsky et al., 2023)](https://arxiv.org/abs/2312.06681). Метод ITI находит головы внимания, связанные с правдивостью, и сдвигает их активации, повышая долю правдивых ответов [(Li et al., 2023)](https://arxiv.org/abs/2306.03341). Общий подход к чтению и управлению концептами вроде честности или эмоций назвали representation engineering [(Zou et al., 2023)](https://arxiv.org/abs/2310.01405).

Самый наглядный результат касается безопасности. [(Arditi et al., 2024)](https://arxiv.org/abs/2406.11717) показали, что отказ отвечать на вредные запросы в 13 открытых чат-моделях опосредован одним направлением. Если вычесть проекцию на него, модель перестаёт отказываться, а если добавить, отказывается даже на безобидные просьбы. Это говорит о том, что выравнивание через дообучение может оказаться довольно тонким слоем поверх базовой модели.

Управлять можно и через признаки SAE: если зафиксировать признак моста Golden Gate на большом значении, Claude начинает считать себя этим мостом [(Templeton et al., 2024)](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html). Векторы персоны (persona vectors) позволяют отслеживать черты вроде угодничества или склонности к галлюцинациям, в том числе в ходе дообучения, и находить обучающие данные, которые эти черты усиливают [(Chen et al., 2025)](https://arxiv.org/abs/2507.21509).

Насколько всё это практично? Бенчмарк AxBench [(Wu et al., 2025)](https://arxiv.org/abs/2501.17148) сравнил методы управления и показал, что обычный промптинг пока выигрывает у методов на основе представлений, а признаки SAE уступают даже простой разности средних. Поэтому управление через представления остаётся скорее исследовательским инструментом и способом проверить гипотезу о признаке, чем заменой промптинга или дообучения.

## Автоматизация интерпретации
В большой модели миллионы признаков, и описать их вручную невозможно. [(Bills et al., 2023)](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) поручили это другой модели: GPT-4 получала тексты с активациями нейрона GPT-2 и писала объяснение. Качество объяснения проверяли симуляцией: модель по одному объяснению предсказывала активации на новых текстах, и корреляция с реальными активациями служила оценкой. Хорошо объяснить удалось лишь небольшую долю нейронов, что неудивительно при полисемантичности. С признаками SAE дело идёт лучше, и сегодня автоматические описания строят для миллионов признаков сразу [(Paulo et al., 2024)](https://arxiv.org/abs/2410.13928).

Следующий шаг — агенты-интерпретаторы, которые сами ставят эксперименты: подбирают входы, вмешиваются в активации и уточняют гипотезы, как это делает MAIA [(Shaham et al., 2024)](https://arxiv.org/abs/2404.14394). Anthropic испытала агентов, которые с помощью инструментов интерпретации проводят аудит моделей и ищут у них скрытые нежелательные цели [(Anthropic, 2025b)](https://alignment.anthropic.com/2025/automated-auditing). Главный риск тот же, что и у людей: объяснение может звучать убедительно и при этом быть неверным [(Huang et al., 2023)](https://arxiv.org/abs/2309.10312). Поэтому автоматические описания тоже нужно проверять вмешательствами.

## Методология и критическое мышление
В интерпретируемости легко найти то, что ищешь. Хорошая иллюстрация — эксперимент [(Jonas & Kording, 2017)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005268). Авторы применили стандартные методы нейробиологии к процессору MOS 6502, на котором запускали старые видеоигры. Методы выдавали правдоподобные результаты, например нашлись транзисторы, удаление которых ломало только Donkey Kong, но ничего не говорили о реальном устройстве процессора.

Типичные ошибки мы уже встречали: объяснение по отобранным примерам, которое не переносится на другой датасет; патчинг подпространств, который «будит» неиспользуемые пути; локализация, которая не подсказывает, где вмешиваться. Отсюда несколько правил. Проверять каждую гипотезу вмешательством, а не только наблюдением. Сравнивать результат со случайными направлениями и случайными компонентами. Формулировать гипотезы так, чтобы они предсказывали поведение на новых входах. И измерять, какую долю поведения объясняет найденный механизм. Для сравнения методов появились бенчмарки: MIB для схем и каузальных переменных [(Mueller et al., 2025)](https://arxiv.org/abs/2504.13151) и SAEBench для автоэнкодеров.

Открытых проблем много [(Sharkey et al., 2025)](https://arxiv.org/abs/2501.16496). Неясно, какая единица анализа правильная: нейрон, направление, признак SAE или что-то ещё. Методы плохо масштабируются, и даже лучшие графы атрибуции объясняют лишь часть вычислений. Мало известно о том, как механизмы формируются в процессе обучения; этим занимается отдельное направление, developmental interpretability [(Hoogland et al., 2024)](https://arxiv.org/abs/2402.02364). И пока нет способа строго доказать, что модель чего-то не делает.

## Применения
Главная мотивация многих исследований — безопасность. В эксперименте [(Marks et al., 2025)](https://arxiv.org/abs/2503.10965) модель намеренно обучили скрытой цели: подстраиваться под известные ошибки модели вознаграждения. Затем несколько команд, не знавших об этом, провели аудит, и команды с доступом к весам и признакам SAE нашли причину странного поведения. Линейные пробы на активациях Llama-3.3-70B распознают стратегический обман, например в сценарии с инсайдерской торговлей [(Goldowsky-Dill et al., 2025)](https://arxiv.org/abs/2502.03407). А векторы персоны из предыдущего раздела позволяют заметить нежелательный сдвиг характера модели ещё во время дообучения.

Интерпретируемость помогает и в отладке. Метод SHIFT из работы о разреженных схемах признаков [(Marks et al., 2024)](https://arxiv.org/abs/2403.19647) нашёл в классификаторе профессий признаки, связанные с полом, и, выключив их, убрал ложную корреляцию без сбора новых данных. Редактирование знаний (ROME и MEMIT [(Meng et al., 2022)](https://arxiv.org/abs/2210.07229)) и удаление опасных знаний, например методом RMU [(Li et al., 2024)](https://arxiv.org/abs/2403.03218), тоже опираются на локализацию информации внутри модели, хотя, как мы видели, связь локализации с удачным вмешательством неочевидна.

Наконец, интерпретируемость может учить самих людей. [(Schut et al., 2023)](https://arxiv.org/abs/2310.16410) извлекли из AlphaZero шахматные концепции, которых нет в человеческой теории игры, и показали, что гроссмейстеры способны им научиться. Похожие работы ведутся на моделях белков и геномов, где модель может знать то, чего ещё не знает биология.

Механистическая интерпретируемость остаётся молодой областью: многие её результаты получены на небольших моделях, а методы меняются каждый год. Но уже сейчас она позволяет задавать вопросы, которые раньше казались невозможными: не только что модель отвечает, но и почему.